# Reducción y Compresión del Dataset de Quejas (CFPB)

Este cuaderno realiza el procesamiento completo para modificar el dataset:
1. **Filtra los datos por fecha** descartando los registros antiguos anteriores al `2023-08-01` (reduciendo el tamaño de 5.7 GB a ~4.5 GB / 15 millones de filas).
2. **Comprime el archivo resultante** a formato `.zip` (~236 MB).
3. **Elimina el CSV temporal** si se desea ahorrar espacio en disco.
4. **Carga y verifica directamente** los datos desde el archivo comprimido usando Pandas.

In [ ]:
import os
import zipfile
import io
import pandas as pd

### Paso 1: Reducción del CSV a ~4.5 GB (Quemando datos anteriores al 01/08/2023)

Procesa el archivo en modo **streaming** (línea por línea) para no desbordar la memoria RAM.
Detecta automáticamente si el archivo `complaints.csv` está descomprimido o lo lee directamente desde `complaints.csv.zip`.

In [ ]:
csv_original = "complaints.csv"
zip_original = "../complaints.csv.zip"
csv_reducido = "complaints_4.5gb.csv"
FECHA_CORTE = "2023-08-01"  # Descarta quejas anteriores a esta fecha

filas_guardadas = 0
bytes_guardados = 0

print("Iniciando filtrado...")

# Determinar origen: CSV descomprimido o lectura directa del ZIP original
if os.path.exists(csv_original):
    f_in = open(csv_original, "r", encoding="utf-8", errors="ignore")
elif os.path.exists(zip_original):
    zf = zipfile.ZipFile(zip_original, "r")
    nombre_csv = [n for n in zf.namelist() if n.endswith(".csv")][0]
    f_in = io.TextIOWrapper(zf.open(nombre_csv), encoding="utf-8", errors="ignore")
else:
    raise FileNotFoundError("No se encontró complaints.csv ni complaints.csv.zip")

with f_in:
    with open(csv_reducido, "w", encoding="utf-8", newline="") as f_out:
        encabezado = f_in.readline()
        f_out.write(encabezado)
        bytes_guardados += len(encabezado.encode("utf-8"))
        
        for linea in f_in:
            if linea[:10] >= FECHA_CORTE:
                f_out.write(linea)
                filas_guardadas += 1
                bytes_guardados += len(linea.encode("utf-8"))

print("Filtrado completado con éxito.")
print(f"Filas conservadas: {filas_guardadas:,}")
print(f"Tamaño generado: {bytes_guardados / (1024**3):.2f} GiB ({bytes_guardados / 1e9:.2f} GB)")

### Paso 2: Supercompresión del archivo reducido a .zip (~236 MB)

Comprime el CSV de 4.5 GB a formato `.zip` con máxima compresión (deflate nivel 9).

In [ ]:
zip_salida = "complaints_4.5gb.csv.zip"

print(f"Comprimiendo {csv_reducido} a {zip_salida}...")
with zipfile.ZipFile(zip_salida, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    zf.write(csv_reducido, arcname=os.path.basename(csv_reducido))

tam_mb = os.path.getsize(zip_salida) / (1024 * 1024)
print(f"Listo. Archivo comprimido generado: {tam_mb:.1f} MB")

### Paso 3: Limpieza del CSV de 4.5 GB descomprimido (Opcional)

Una vez que el archivo comprimido `.zip` está generado y verificado, se puede borrar el CSV temporal de 4.5 GB para ahorrar espacio en disco.

In [ ]:
# Eliminar el CSV intermedio de 4.5 GB si ya existe el .zip
if os.path.exists(csv_reducido) and os.path.exists(zip_salida):
    os.remove(csv_reducido)
    print(f"Archivo temporal {csv_reducido} eliminado. Espacio liberado.")

### Paso 4: Carga y verificación directa desde el .zip con Pandas

Pandas puede leer archivos comprimidos directamente sin necesidad de descomprimirlos antes en el disco.

In [ ]:
# Muestra rápida de las primeras 5 filas
df_muestra = pd.read_csv("complaints_4.5gb.csv.zip", nrows=5)
df_muestra

In [ ]:
# Verificación del rango de fechas y conteo total de filas leyendo solo una columna
fechas = pd.read_csv("complaints_4.5gb.csv.zip", usecols=["Date received"])
print(f"Fecha mínima: {fechas['Date received'].min()}")
print(f"Fecha máxima: {fechas['Date received'].max()}")
print(f"Total de quejas en el archivo comprimido: {len(fechas):,}")